# U23AI035 - LAB 8 - RAI

In [32]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [33]:
data_dir = 'Clothes_Dataset'

In [34]:
data = []
for label in os.listdir(data_dir):
    class_path = os.path.join(data_dir, label)
    if os.path.isdir(class_path):
        for img in os.listdir(class_path):
            data.append([os.path.join(class_path, img), label])

df = pd.DataFrame(data, columns=["image", "label"])

In [35]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])

In [36]:
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

In [37]:
print(len(train_df), len(val_df), len(test_df))

6000 750 750


In [38]:
class ClothesDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, "image"]
        label = self.df.loc[idx, "label"]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [39]:
baseline_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [40]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomAffine(degrees=20, translate=(0.1, 0.1)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop(224),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor()
])

In [41]:
val_test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor()
])

In [42]:
batch_size = 32

# Baseline datasets
train_base = ClothesDataset(train_df, baseline_transform)
val_base = ClothesDataset(val_df, baseline_transform)
test_base = ClothesDataset(test_df, baseline_transform)

# Augmented datasets
train_aug = ClothesDataset(train_df, train_transform)
val_aug = ClothesDataset(val_df, val_test_transform)
test_aug = ClothesDataset(test_df, val_test_transform)

train_loader_base = DataLoader(train_base, batch_size=batch_size, shuffle=True)
val_loader_base = DataLoader(val_base, batch_size=batch_size)
test_loader_base = DataLoader(test_base, batch_size=batch_size)

train_loader_aug = DataLoader(train_aug, batch_size=batch_size, shuffle=True)
val_loader_aug = DataLoader(val_aug, batch_size=batch_size)
test_loader_aug = DataLoader(test_aug, batch_size=batch_size)

In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_model(model, train_loader, val_loader, epochs=20):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                _, preds = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (preds == labels).sum().item()

        val_acc = 100 * correct / total

        print(f"Epoch {epoch+1}, Loss: {train_loss:.4f}, Val Acc: {val_acc:.2f}%")

    return model

In [44]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

    return 100 * correct / total

In [45]:
num_classes = len(df["label"].unique())

model_base = models.resnet18(pretrained=True)
for param in model_base.parameters():
    param.requires_grad = False
model_base.fc = nn.Linear(model_base.fc.in_features, num_classes)
model_base = train_model(model_base, train_loader_base, val_loader_base)

test_acc_base = evaluate(model_base, test_loader_base)
print("Baseline Test Accuracy:", test_acc_base)

c:\Users\JAIVAL CHAUHAN\.conda\envs\env-pt\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\JAIVAL CHAUHAN\.conda\envs\env-pt\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1, Loss: 501.4434, Val Acc: 21.73%
Epoch 2, Loss: 436.1975, Val Acc: 39.60%
Epoch 3, Loss: 387.9198, Val Acc: 45.60%
Epoch 4, Loss: 353.0281, Val Acc: 50.27%
Epoch 5, Loss: 325.8718, Val Acc: 52.93%
Epoch 6, Loss: 305.4576, Val Acc: 55.60%
Epoch 7, Loss: 289.0141, Val Acc: 57.20%
Epoch 8, Loss: 275.0014, Val Acc: 57.60%
Epoch 9, Loss: 264.2101, Val Acc: 59.33%
Epoch 10, Loss: 254.5726, Val Acc: 59.07%
Epoch 11, Loss: 246.4134, Val Acc: 58.93%
Epoch 12, Loss: 239.4853, Val Acc: 60.93%
Epoch 13, Loss: 234.3598, Val Acc: 60.93%
Epoch 14, Loss: 228.5691, Val Acc: 61.73%
Epoch 15, Loss: 223.7383, Val Acc: 61.33%
Epoch 16, Loss: 218.7344, Val Acc: 61.60%
Epoch 17, Loss: 214.1962, Val Acc: 62.13%
Epoch 18, Loss: 211.4100, Val Acc: 62.67%
Epoch 19, Loss: 208.7315, Val Acc: 62.80%
Epoch 20, Loss: 204.9336, Val Acc: 63.33%
Baseline Test Accuracy: 66.4


In [ ]:
model_aug = models.resnet18(pretrained=True)
for param in model_aug.parameters():
    param.requires_grad = False
model_aug.fc = nn.Linear(model_aug.fc.in_features, num_classes)
model_aug = train_model(model_aug, train_loader_aug, val_loader_aug)

test_acc_aug = evaluate(model_aug, test_loader_aug)
print("Augmented Test Accuracy:", test_acc_aug)

Epoch 1, Loss: 510.3211, Val Acc: 14.93%
Epoch 2, Loss: 470.8316, Val Acc: 29.87%
Epoch 3, Loss: 443.4340, Val Acc: 39.20%
Epoch 4, Loss: 417.9688, Val Acc: 41.87%
Epoch 5, Loss: 399.2509, Val Acc: 49.33%
Epoch 6, Loss: 384.1008, Val Acc: 50.27%
Epoch 7, Loss: 372.0767, Val Acc: 52.13%
Epoch 8, Loss: 361.3856, Val Acc: 53.60%
Epoch 9, Loss: 353.2169, Val Acc: 54.67%
Epoch 10, Loss: 346.6204, Val Acc: 54.00%
Epoch 11, Loss: 340.4130, Val Acc: 55.07%
Epoch 12, Loss: 335.3366, Val Acc: 56.00%
Epoch 13, Loss: 330.9012, Val Acc: 56.40%
Epoch 14, Loss: 323.5575, Val Acc: 56.80%
Epoch 15, Loss: 320.6890, Val Acc: 57.20%
Epoch 16, Loss: 316.5089, Val Acc: 56.27%
Epoch 17, Loss: 315.1505, Val Acc: 57.60%
Epoch 18, Loss: 313.4530, Val Acc: 58.13%
Epoch 19, Loss: 309.0761, Val Acc: 56.80%
Epoch 20, Loss: 305.6324, Val Acc: 57.87%
Augmented Test Accuracy: 60.666666666666664


#### Original Dataset Accuracy: 66.4% <BR> Augmented Dataset Accuracy: 60.67%

### How to build robus models?
#### Use diverse augmentations, use different augmentations in test then in train, use transfer learning, increase dataset diversity